### Kisan Call Centre transcripts — exploratory notebook

- **Scope**: Farmer query transcripts across selected districts in **Uttar Pradesh** and **Madhya Pradesh** (combined into one dataframe `df`).
- **Goal**: Profile the dataset (shape/missingness), then summarize dominant crops and seasonal crop signals—without assuming perfect labeling.


### Load transcript CSV

- **Purpose**: Load one combined local CSV into `df` for analysis.
- **Source**: `MP__UP_and_Raj.csv` in the same folder as this notebook.
- **Checks**: print shape, state distribution, and column names before analysis.


In [ ]:
import pandas as pd

df = pd.read_csv("MP, UP and Raj.csv")
# Quick sanity check: rows and columns in the loaded dataset
print(df.shape)
print(df["StateName"].value_counts())
print(list(df.columns))


(18081, 11)
StateName
RAJASTHAN         10796
UTTAR PRADESH      4588
MADHYA PRADESH     2696
Name: count, dtype: int64
['Season', 'Sector', 'Category', 'Crop', 'QueryType', 'QueryText', 'KccAns', 'StateName', 'DistrictName', 'BlockName', 'CreatedOn']


### Initial dataframe inspection


In [12]:
# Question answered: What do the first few transcript records look like in context?
# What this tells us: We can quickly verify whether columns look sensible,
# spot obvious data-entry quirks, and confirm rows are aligned before deeper analysis.

df.head()


,Season,Sector,Category,Crop,QueryType,QueryText,KccAns,StateName,DistrictName,BlockName,CreatedOn
0,NaN,AGRICULTURE,Oilseeds,Groundnut (pea nut/mung phalli),\tPlant Protection\t,Farmer wants to know information about How to ...,Recommended for to control of fungal attack us...,MADHYA PRADESH,BARWANI,SENDHAWA,2019-02-01T11:39:53.497
1,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to know information about Jai Ki...,"Dear farmer, to avail the Jai kisan crop loan ...",MADHYA PRADESH,BARWANI,SENDHAWA,2019-02-02T09:09:41.27
2,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to know information about agricul...,To know information about agriculture scheme t...,MADHYA PRADESH,BARWANI,SENDHAWA,2019-02-02T09:11:05.183
3,NaN,HORTICULTURE,Fruits,Melon,Vegetative Propagation and Tissue Culture,farmer want to know information about varietie...,recommended for varieties of water melon Sugar...,MADHYA PRADESH,BARWANI,RAJPUR,2019-02-02T13:15:18.097
4,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to start mandi rate facility on h...,Your mandi rate facility registration has been...,MADHYA PRADESH,BARWANI,RAJPUR,2019-02-02T14:08:01.317


In [14]:
# Question answered: What is the structural health of this dataset?
# What this tells us: It reveals row count, column dtypes, and non-null counts,
# so we can judge data completeness and whether type conversion/cleaning is needed.

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18081 entries, 0 to 18080
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Season        3 non-null      object
 1   Sector        18080 non-null  object
 2   Category      18080 non-null  object
 3   Crop          18080 non-null  object
 4   QueryType     18080 non-null  object
 5   QueryText     18080 non-null  object
 6   KccAns        9310 non-null   object
 7   StateName     18080 non-null  object
 8   DistrictName  18080 non-null  object
 9   BlockName     18080 non-null  object
 10  CreatedOn     18080 non-null  object
dtypes: object(11)
memory usage: 1.5+ MB


In [15]:
# Question answered: Which values dominate each column and how varied is the data?
# What this tells us: We get a high-level profile of frequencies/uniques/top values,
# which highlights dominant labels and helps frame what patterns are worth exploring next.

df.describe(include="all")


,Season,Sector,Category,Crop,QueryType,QueryText,KccAns,StateName,DistrictName,BlockName,CreatedOn
count,3,18080,18080,18080,18080,18080,9310,18080,18080,18080,18080
unique,3,4,22,175,56,8577,3656,3,19,153,17924
top,RABI,AGRICULTURE,Others,Others,Weather,TELL ME ABOUT WEATHER INFORMATION ?\n,NO RAIN POSSIBILITY IN NEXT 5 DAYS BUT CLOUDY...,RAJASTHAN,BIKANER,JAISALMER,2019-02-11T12:30:52.19
freq,1,14543,10944,10705,8233,2319,553,10796,2701,1368,12


### Missing values


In [16]:
# Question answered: Where is information missing across columns?
# What this tells us: Missing counts show which fields may weaken interpretations;
# for example, missing Crop/Season/QueryType can limit crop-cycle comparisons.

df.isnull().sum()


Season          18078
Sector              1
Category            1
Crop                1
QueryType           1
QueryText           1
KccAns           8771
StateName           1
DistrictName        1
BlockName           1
CreatedOn           1
dtype: int64

### Most common crops overall


In [17]:
# Question answered: Which crop labels appear most frequently in farmer queries?
# What this tells us: The top counts indicate which crop topics drive advisory demand
# in this dataset, while labels like "Others" suggest many queries are broad/non-specific.

df["Crop"].value_counts().head(15)


Crop
Others                                       10705
Wheat                                         1375
Cumin                                          805
Bengal Gram (Gram/Chick Pea/Kabuli/Chana)      757
0                                              366
Sugarcane (Noble Cane)                         366
Mustard                                        288
Onion                                          223
Mango                                          193
Garlic                                         189
Tomato                                         158
Chillies                                       128
Citrus                                         117
Cotton (Kapas)                                 108
Bovine(Cow,Buffalo)                             98
Name: count, dtype: int64

### Focus on the single most common crop label

- **Purpose**: Zoom into the dominant `Crop` category from the frequency table (whatever ranks #1 overall).
- **Input**: `df`, filtered with boolean indexing on `Crop`.
- **Output**: How many rows match + `head(10)` examples for qualitative spot-checking.
- **Caveat**: If `#1` is a broad bucket like `"Others"`, you’re mostly seeing general/non-specific routing—not one botanical crop.


In [ ]:
# Identify the most frequently mentioned crop in the dataset
top_crop = df["Crop"].value_counts().idxmax()
crop_subset = df[df["Crop"] == top_crop]

# Display the most frequently mentioned crop
print(top_crop)
print(len(crop_subset))
crop_subset.head(10)


Others
10705


,Season,Sector,Category,Crop,QueryType,QueryText,KccAns,StateName,DistrictName,BlockName,CreatedOn
1,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to know information about Jai Ki...,"Dear farmer, to avail the Jai kisan crop loan ...",MADHYA PRADESH,BARWANI,SENDHAWA,2019-02-02T09:09:41.27
2,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to know information about agricul...,To know information about agriculture scheme t...,MADHYA PRADESH,BARWANI,SENDHAWA,2019-02-02T09:11:05.183
4,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to start mandi rate facility on h...,Your mandi rate facility registration has been...,MADHYA PRADESH,BARWANI,RAJPUR,2019-02-02T14:08:01.317
5,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer wants to know information about govt. S...,Recommended for to know about govt. Schemes pl...,MADHYA PRADESH,BARWANI,RAJPUR,2019-02-02T15:02:02.473
7,NaN,AGRICULTURE,Others,Others,Market Information,Farmer want to know information about mandi ra...,Recommended mandi rate of Onion crop in Indore...,MADHYA PRADESH,BARWANI,RAJPUR,2019-02-03T08:36:25.33
8,NaN,AGRICULTURE,Others,Others,Weather,Farmer needs information regarding for barwani...,According to meteorology department of India t...,MADHYA PRADESH,BARWANI,BARWANI,2019-02-03T09:13:12.33
9,NaN,AGRICULTURE,Others,Others,Weather,OUT OF STATE CALL\n,OUT OF STATE CALL\n,MADHYA PRADESH,BARWANI,0,2019-02-03T11:15:41.73
10,NaN,AGRICULTURE,Others,Others,Weather,OUT OF STATE CALL\n,OUT OF STATE CALL\n,MADHYA PRADESH,BARWANI,0,2019-02-03T11:18:57.527
13,NaN,AGRICULTURE,Others,Others,Government Schemes,Farmer want to know information about govt. sc...,Recommended for to know about govt. schemes pl...,MADHYA PRADESH,BARWANI,BARWANI,2019-02-04T19:07:00.19
14,NaN,AGRICULTURE,Others,Others,Training and Exposure Visits,farmer want to know information about agricul...,recommended for to know information about agri...,MADHYA PRADESH,BARWANI,PANSEMAL,2019-02-04T21:07:56.67


### Queries by calendar month and crop (from `CreatedOn`)

- **Purpose**: Rank crops by **how often they appear in transcripts within each calendar month**, based on when the call was logged (`CreatedOn`).
- **Why not `Season` here**: In this extract, `Season` is almost entirely blank, so it cannot reliably represent farming cycles.
- **How to read**: Within each month, larger counts mean more labeled transcripts mentioning that `Crop` during that month’s calls (a **call-volume / timing proxy**, not the same as agronomic sowing/harvest calendars unless you define explicit crop calendars).
- **Label note**: `Others` means **not mapped to one specific crop** (a catch‑all bucket—often general agriculture / schemes / markets-type queries).
- **Caveat**: This still depends on transcript labeling, and rows with unparseable timestamps show up under a missing month (`NaT`) at the bottom after sorting.


In [ ]:
created_dt = pd.to_datetime(df["CreatedOn"], errors="coerce")

created_month = created_dt.dt.to_period("M")

# Group by month and crop, count occurrences
crop_queries_by_month = (
    df.assign(created_month=created_month)
    .groupby(["created_month", "Crop"], dropna=False)
    .size()
    .reset_index(name="query_count")
    .sort_values(
        ["created_month", "query_count", "Crop"],
        ascending=[True, False, True],
        na_position="last",
    )
)

print(
    'Note: In this dataset, Crop == "Others" means the transcript was labeled as not mapped '
    "to one specific crop (a catch‑all bucket), often general agriculture/scheme/market-type queries."
)

crop_queries_by_month


Note: In this dataset, Crop == "Others" means the transcript was labeled as not mapped to one specific crop (a catch‑all bucket), often general agriculture/scheme/market-type queries.


,created_month,Crop,query_count
85,2019-01,Others,4747
121,2019-01,Wheat,827
13,2019-01,Bengal Gram (Gram/Chick Pea/Kabuli/Chana),513
40,2019-01,Cumin,219
79,2019-01,Mustard,192
...,...,...,...
600,NaT,Papaya,1
601,NaT,Pumpkin,1
602,NaT,Rocket salad (taramira),1
603,NaT,Rose,1


### Assignment questions

This section answers the three required analysis questions about farmer support demand patterns in the 2019 KCC transcript data.


In [21]:
# Question answered: What issue categories (QueryType) are farmers asking about most?
# What this tells us: The highest counts show which support needs dominate call traffic,
# helping identify where advisory resources and response scripts are most needed.

df["QueryType"].value_counts().head(15)


QueryType
Weather                                      8233
\tPlant Protection\t                         3612
Government Schemes                           1714
Nutrient Management                          1471
Market Information                            759
Cultural Practices                            393
Fertilizer Use and Availability               288
Weed Management                               233
Varieties                                     213
Seeds and Planting Material                   173
Soil Testing                                  109
Vegetative Propagation and Tissue Culture      81
Crop Insurance                                 80
Training and Exposure Visits                   66
\tWater Management\t                           63
Name: count, dtype: int64

In [22]:
# Question answered: Which crop categories are farmers asking about most often overall?
# What this tells us: High-frequency crops indicate where farmer demand clusters,
# helping prioritize crop-specific advisories and extension support.

df["Crop"].value_counts().head(15)


Crop
Others                                       10705
Wheat                                         1375
Cumin                                          805
Bengal Gram (Gram/Chick Pea/Kabuli/Chana)      757
0                                              366
Sugarcane (Noble Cane)                         366
Mustard                                        288
Onion                                          223
Mango                                          193
Garlic                                         189
Tomato                                         158
Chillies                                       128
Citrus                                         117
Cotton (Kapas)                                 108
Bovine(Cow,Buffalo)                             98
Name: count, dtype: int64

In [23]:
# Question answered: Which specific QueryType × Crop combinations occur most often?
# What this tells us: Pair-level ranking reveals concrete problem contexts (for example,
# a specific issue type repeatedly tied to a crop), which is useful for targeted support planning.

querytype_crop_counts = (
    df.groupby(["QueryType", "Crop"], dropna=False)
    .size()
    .reset_index(name="query_count")
    .sort_values(["query_count", "QueryType", "Crop"], ascending=[False, True, True])
)

querytype_crop_counts.head(20)


,QueryType,Crop,query_count
767,Weather,Others,7947
384,Government Schemes,Others,1597
47,\tPlant Protection\t,Cumin,542
424,Market Information,Others,528
25,\tPlant Protection\t,Bengal Gram (Gram/Chick Pea/Kabuli/Chana),511
127,\tPlant Protection\t,Wheat,466
547,Nutrient Management,Wheat,429
753,Weather,0,247
87,\tPlant Protection\t,Mustard,178
489,Nutrient Management,Cumin,169


In [25]:
# Question answered: How many queries were effectively unanswered by KCC?
# What this tells us: The unanswered share estimates potential service gaps where farmers
# did not receive a recorded response and may need follow-up support.

unanswered_mask = df["KccAns"].isna() | (df["KccAns"].astype(str).str.strip() == "")
unanswered_df = df[unanswered_mask].copy()

unanswered_count = len(unanswered_df)
total_count = len(df)
unanswered_pct = (unanswered_count / total_count) * 100

print(f"Unanswered queries: {unanswered_count} / {total_count} ({unanswered_pct:.2f}%)")


Unanswered queries: 8771 / 18081 (48.51%)


In [26]:
# Question answered: Which crop contexts are most common among unanswered queries?
# What this tells us: Repeated unanswered calls for particular crops can indicate where
# farmers may face unresolved advisory needs and risk of delayed decisions.

unanswered_df["Crop"].value_counts().head(15)


Crop
Others                                       5175
Cumin                                         580
Sugarcane (Noble Cane)                        355
Wheat                                         316
Mango                                         164
0                                             162
Bengal Gram (Gram/Chick Pea/Kabuli/Chana)     155
Cotton (Kapas)                                106
Tomato                                         98
Chillies                                       91
Mustard                                        83
Citrus                                         70
Bhindi(Okra/Ladysfinger)                       67
Bottle Gourd                                   60
Onion                                          60
Name: count, dtype: int64

In [27]:
# Question answered: What topics (QueryType) are most common among unanswered queries?
# What this tells us: If certain topics dominate unanswered cases, those areas likely
# need stronger response workflows, clearer scripts, or better staffing.

unanswered_df["QueryType"].value_counts().head(15)


QueryType
Weather                                      3753
\tPlant Protection\t                         1748
Government Schemes                            991
Nutrient Management                           578
Market Information                            443
Cultural Practices                            237
Varieties                                     182
Fertilizer Use and Availability               133
Weed Management                               107
Seeds and Planting Material                    95
Crop Insurance                                 44
Vegetative Propagation and Tissue Culture      42
Seeds                                          41
Sowing Time and Weather                        40
\tWater Management\t                           33
Name: count, dtype: int64

In [28]:
# Question answered: What do unanswered queries actually look like in plain text?
# What this tells us: Reading a sample helps validate whether missing answers reflect
# true service gaps, logging issues, or specific recurring request patterns.

unanswered_df[["StateName", "QueryType", "Crop", "QueryText"]].head(10)


,StateName,QueryType,Crop,QueryText
61,MADHYA PRADESH,Government Schemes,Others,Farmer want to know information about in solar...
62,MADHYA PRADESH,Government Schemes,Others,Farmer want to know information about govt. sc...
63,MADHYA PRADESH,Government Schemes,Others,Farmer want to know information about govt. sc...
64,MADHYA PRADESH,\tPlant Protection\t,Watermelon,Farmer want to know how to white fly control ...
65,MADHYA PRADESH,Government Schemes,Others,\nFarmer want to know information about govt. ...
66,MADHYA PRADESH,Government Schemes,Others,Farmer want to know information about govt. sc...
67,MADHYA PRADESH,Government Schemes,Others,Farmer want to know information about govt. sc...
68,MADHYA PRADESH,Market Information,Periwinkle,farmer want to know about Quinoa crop?
69,MADHYA PRADESH,\tField Preparation\t,Others,farmer want to know information about complant...
70,MADHYA PRADESH,Weather,Others,Farmer needs information regarding weather for...


In [29]:
# Question answered: Do states differ in the mix of query topics they call about?
# What this tells us: Raw counts show the volume pattern by state and topic,
# but larger states can dominate totals, so percentages are needed next for fair comparison.

state_querytype_counts = pd.crosstab(df["StateName"], df["QueryType"])
state_querytype_counts


QueryType,\tField Preparation\t,\tPlant Protection\t,\tWater Management\t,0,Agriculture Mechanization,Animal Breeding,Animal Nutrition,"Animal Production (Piggery, Goatery, Sheep Farming etc.)",Artificial Insemination,Bio-Pesticides and Bio-Fertilizers,...,Sowing Time and Weather,Spices and Condiment Crops,Storage,"Tank, Pond and Reservoir Management",Training and Exposure Visits,Varieties,Vegetative Propagation and Tissue Culture,"Water Management, Micro Irrigation",Weather,Weed Management
StateName,,,,,,,,,,,,,,,,,,,,,
MADHYA PRADESH,29,682,28,0,6,2,2,0,0,4,...,12,0,0,2,8,46,11,4,547,27
RAJASTHAN,20,2142,13,6,14,13,2,7,1,1,...,14,11,2,2,26,129,50,15,5237,102
UTTAR PRADESH,9,788,22,28,29,4,3,8,1,5,...,23,5,3,0,32,38,20,7,2449,104


In [30]:
# Question answered: Within each state, which query topics take the largest share?
# What this tells us: Percentage shares remove state size effects, so differences reflect
# topic mix rather than just higher call volume in larger states.

state_querytype_pct = pd.crosstab(df["StateName"], df["QueryType"], normalize="index") * 100
state_querytype_pct.round(2)


QueryType,\tField Preparation\t,\tPlant Protection\t,\tWater Management\t,0,Agriculture Mechanization,Animal Breeding,Animal Nutrition,"Animal Production (Piggery, Goatery, Sheep Farming etc.)",Artificial Insemination,Bio-Pesticides and Bio-Fertilizers,...,Sowing Time and Weather,Spices and Condiment Crops,Storage,"Tank, Pond and Reservoir Management",Training and Exposure Visits,Varieties,Vegetative Propagation and Tissue Culture,"Water Management, Micro Irrigation",Weather,Weed Management
StateName,,,,,,,,,,,,,,,,,,,,,
MADHYA PRADESH,1.08,25.30,1.04,0.00,0.22,0.07,0.07,0.00,0.00,0.15,...,0.45,0.00,0.00,0.07,0.30,1.71,0.41,0.15,20.29,1.00
RAJASTHAN,0.19,19.84,0.12,0.06,0.13,0.12,0.02,0.06,0.01,0.01,...,0.13,0.10,0.02,0.02,0.24,1.19,0.46,0.14,48.51,0.94
UTTAR PRADESH,0.20,17.18,0.48,0.61,0.63,0.09,0.07,0.17,0.02,0.11,...,0.50,0.11,0.07,0.00,0.70,0.83,0.44,0.15,53.38,2.27


In [31]:
# Question answered: What are the top query topics in each state specifically?
# What this tells us: This compact ranking highlights where support concerns diverge
# by state, guiding more localized advisory planning and resource allocation.

top_querytypes_per_state = (
    df.groupby(["StateName", "QueryType"], dropna=False)
    .size()
    .reset_index(name="query_count")
    .sort_values(["StateName", "query_count", "QueryType"], ascending=[True, False, True])
    .groupby("StateName", group_keys=False)
    .head(10)
)

top_querytypes_per_state


,StateName,QueryType,query_count
1,MADHYA PRADESH,\tPlant Protection\t,682
17,MADHYA PRADESH,Government Schemes,667
35,MADHYA PRADESH,Weather,547
21,MADHYA PRADESH,Nutrient Management,198
14,MADHYA PRADESH,Fertilizer Use and Availability,101
10,MADHYA PRADESH,Cultural Practices,100
19,MADHYA PRADESH,Market Information,85
32,MADHYA PRADESH,Varieties,46
26,MADHYA PRADESH,Seeds and Planting Material,39
9,MADHYA PRADESH,Crop Insurance,31
